# Matrix Operations

| Difficulty | ██░░░░░░░░ 2/10 |
| :--- | :--- |
| Prerequisites | 01.01 |
| Time | ~60 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/02_matrix_operations.ipynb)

---

## 🎯 Learning Objective

Learn the core matrix operations — multiplication, transpose, and rank — and the four fundamental subspaces associated with a matrix.

---

## 📐 Theory

### Matrix multiplication

Given $A$ of shape $(m,n)$ and $B$ of shape $(n,p)$, their product $C = AB$ has shape $(m,p)$
with entries

$$C_{ik} = \sum_{j=1}^{n} A_{ij}B_{jk}$$

Entry $(i,k)$ of the product is the dot product of row $i$ of $A$ with column $k$ of $B$ —
matrix multiplication is nothing but a grid of dot products.

### Transpose

The transpose $A^\top$ flips rows and columns: $(A^\top)_{ij} = A_{ji}$. A useful identity
that comes up constantly when deriving gradients: $(AB)^\top = B^\top A^\top$ (the order
reverses).

### Rank and the four fundamental subspaces

The **column space** of $A$ is the span of its columns — every possible output $A\mathbf{x}$.
The **row space** is the span of its rows. The **rank** of $A$ is the dimension of the column
space, which always equals the dimension of the row space (a genuinely non-obvious fact). The
**null space** is $\{\mathbf{x} \mid A\mathbf{x} = \mathbf{0}\}$ — every input that gets
crushed to zero.

These are tied together by the **rank-nullity theorem**: for an $A$ with $n$ columns,

$$\text{rank}(A) + \dim(\text{null}(A)) = n$$

Intuitively: every input dimension either survives into the output (contributing to rank) or
gets annihilated (contributing to the null space) — there's nowhere else for it to go.

---

## 👁️ Visual Intuition

Applying a matrix to a whole grid of points shows rank directly: a full-rank $2\times 2$
matrix keeps the grid two-dimensional, while a rank-deficient matrix squashes it onto a line.

In [ ]:
# Rank made visible: a full-rank matrix keeps the grid 2D, a rank-deficient one flattens it
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

xs, ys = np.meshgrid(np.linspace(-1, 1, 15), np.linspace(-1, 1, 15))
grid = np.stack([xs.ravel(), ys.ravel()])   # shape (2, 225)

A_full = np.array([[2.0, 1.0], [0.0, 1.5]])          # full rank: columns point in different directions
A_deficient = np.array([[1.0, 2.0], [0.5, 1.0]])      # row 2 = 0.5 * row 1 -> rank 1

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, A, title in zip(axes, [A_full, A_deficient],
                         ["Full rank: stays 2D", "Rank-deficient: collapses to a line"]):
    transformed = A @ grid
    ax.scatter(transformed[0], transformed[1], s=8, color="#4C72B0")
    ax.set_title(f"{title}\nrank = {np.linalg.matrix_rank(A)}")
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 🐍 Implementation from Scratch

Matrix multiplication from three nested loops makes the summation formula concrete. Then we
verify the rank-nullity theorem computationally on a matrix with genuine rank deficiency.

In [ ]:
# Matrix multiplication from scratch, plus a rank-nullity theorem sanity check
import numpy as np
from scipy.linalg import null_space

def matmul_from_scratch(A, B):
    m, n = A.shape
    n2, p = B.shape
    assert n == n2, "inner dimensions must match"
    C = np.zeros((m, p))
    for i in range(m):
        for k in range(p):
            total = 0.0
            for j in range(n):
                total += A[i, j] * B[j, k]
            C[i, k] = total
    return C

A = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
B = np.array([[7.0, 8.0], [9.0, 10.0], [11.0, 12.0]])
print("matmul_from_scratch matches A @ B:", np.allclose(matmul_from_scratch(A, B), A @ B))
print("(AB)^T == B^T A^T:                ", np.allclose((A @ B).T, B.T @ A.T))

M = np.array([[1.0, 2.0, 3.0], [2.0, 4.0, 6.0], [1.0, 0.0, 1.0]])   # row 2 = 2 * row 1 -> rank-deficient
rank = np.linalg.matrix_rank(M)
ns = null_space(M)
print(f"\nrank(M) = {rank},  dim(null(M)) = {ns.shape[1]},  n (columns) = {M.shape[1]}")
print("Rank-nullity theorem holds:", rank + ns.shape[1] == M.shape[1])

## 🤖 Why This Matters for AI

Every dense layer in a neural network computes $\mathbf{y} = W\mathbf{x} + \mathbf{b}$ — the
weight matrix $W$ **is** a learned linear transformation. The rank of $W$ caps how much of the
input's information can possibly survive: a rank-$k$ weight matrix can only produce outputs
that are combinations of $k$ fixed directions, no matter what the input is. This is precisely
the bottleneck that low-rank adaptation methods like LoRA exploit deliberately — a preview of
[Notebook 01.07](07_SVD_and_PCA.ipynb).

In [ ]:
# A rank-2 weight matrix caps EVERY possible output to a 2D subspace, regardless of input
import numpy as np

rng = np.random.default_rng(0)
d_in, d_out = 6, 6

W_full = rng.standard_normal((d_out, d_in))
print("Full random weight matrix rank:", np.linalg.matrix_rank(W_full), "out of", min(d_in, d_out))

# Force a rank-2 weight matrix, built as a sum of two rank-1 outer products
u1, u2 = rng.standard_normal(d_out), rng.standard_normal(d_out)
v1, v2 = rng.standard_normal(d_in), rng.standard_normal(d_in)
W_low_rank = np.outer(u1, v1) + np.outer(u2, v2)
print("Constrained weight matrix rank:", np.linalg.matrix_rank(W_low_rank))

xs = rng.standard_normal((10, d_in))          # 10 different random inputs
outputs = xs @ W_low_rank.T                    # every one of the 10 outputs...
recovered_rank = np.linalg.matrix_rank(outputs)
print("\nRank of 10 stacked outputs from W_low_rank:", recovered_rank)
print("No matter which/how many inputs you try, outputs never escape a", recovered_rank, "-dimensional subspace.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Multiply $\begin{pmatrix}1&2\\3&4\end{pmatrix}$ by $\begin{pmatrix}5&6\\7&8\end{pmatrix}$ by
   hand using the summation formula, then verify with `matmul_from_scratch`. Before multiplying,
   predict the shape of the result from the two input shapes alone.

<details>
<summary>💡 Solution</summary>

Both inputs are $(2,2)$, so the shape rule $(m,n)\times(n,p)\to(m,p)$ predicts a $(2,2)$ result.
Entry $(1,1)$: $1\cdot5+2\cdot7=19$. Entry $(1,2)$: $1\cdot6+2\cdot8=22$. Entry $(2,1)$:
$3\cdot5+4\cdot7=43$. Entry $(2,2)$: $3\cdot6+4\cdot8=50$. So
$$\begin{pmatrix}1&2\\3&4\end{pmatrix}\begin{pmatrix}5&6\\7&8\end{pmatrix} =
\begin{pmatrix}19&22\\43&50\end{pmatrix},$$
shape $(2,2)$ as predicted, matching `matmul_from_scratch(A, B)` exactly.

</details>

### 💭 UNDERSTAND
2. For the same two matrices $A=\begin{pmatrix}1&2\\3&4\end{pmatrix}$,
   $B=\begin{pmatrix}5&6\\7&8\end{pmatrix}$, predict — without computing — whether $AB$ and $BA$
   will be equal. Compute both to check, then explain what property of the summation formula
   $C_{ik}=\sum_jA_{ij}B_{jk}$ makes matrix multiplication order-dependent in general (unlike
   multiplying two ordinary numbers).

<details>
<summary>💡 Solution</summary>

$AB=\begin{pmatrix}19&22\\43&50\end{pmatrix}$ but
$BA=\begin{pmatrix}23&34\\31&46\end{pmatrix}$ — different matrices, so $AB\ne BA$ here. The root
cause: $C_{ik}=\sum_jA_{ij}B_{jk}$ pairs row $i$ of the *first* factor with column $k$ of the
*second* factor — swapping the order swaps which matrix contributes rows and which contributes
columns to every single entry, and nothing in the formula guarantees those two different pairings
produce the same numbers. Ordinary number multiplication has no such row/column asymmetry to
exploit (a number has no "rows" or "columns"), which is why commutativity is automatic there but
not here. (Two special cases where it does still commute: $A$ with the identity, or $A$ with
itself.)

</details>

### ✏️ DERIVE
3. Derive the transpose identity $(AB)^\top = B^\top A^\top$ stated in the Theory section,
   starting from the entrywise definitions $(AB)_{ik}=\sum_jA_{ij}B_{jk}$ and
   $(X^\top)_{ij}=X_{ji}$. Show the identity holds entry-by-entry, not just "in general."

<details>
<summary>💡 Solution outline</summary>

Fix output indices $(i,k)$ and track what each side gives for the transposed product's entry
$(k,i)$ (since $(AB)^\top$ has its rows/columns swapped relative to $AB$):
$$\big[(AB)^\top\big]_{ki} = (AB)_{ik} = \sum_{j} A_{ij}B_{jk}.$$
Now expand the right-hand side the same way, using $(A^\top)_{ji}=A_{ij}$ and
$(B^\top)_{kj}=B_{jk}$:
$$\big[B^\top A^\top\big]_{ki} = \sum_j (B^\top)_{kj}(A^\top)_{ji} = \sum_j B_{jk}A_{ij}.$$
Both sums are over the same index $j$ and multiply the same two scalars $A_{ij}$ and $B_{jk}$
(just written in a different order — scalar multiplication *does* commute even though matrix
multiplication doesn't), so the two sums are identical term-by-term for every choice of $(i,k)$.
Since $(i,k)$ was arbitrary, $(AB)^\top=B^\top A^\top$ holds entry-by-entry, confirmed numerically
above with `(A @ B).T` vs `B.T @ A.T` matching to floating-point precision for the notebook's own
non-square $A,B$.

</details>

### 🐍 IMPLEMENT
4. Write a function `verify_rank_nullity(M)` that computes `np.linalg.matrix_rank(M)` and
   `scipy.linalg.null_space(M).shape[1]`, and returns whether their sum equals `M.shape[1]`.
   Test it on the notebook's own rank-deficient `M`, on the identity matrix, and on a random
   (generically full-rank) non-square matrix.

<details>
<summary>✅ How to know you're right</summary>

`verify_rank_nullity` must return `True` on all three test cases — the rank-nullity theorem is
not an approximation that holds "usually," it's an exact equality for *every* matrix, so there
is no valid input on which a correctly-implemented checker should return `False`. As an
additional independent check (not just trusting the two library calls to agree with each other
by coincidence), pick any vector from the returned null space basis and confirm `M @ v` is
approximately the zero vector directly — this is the actual defining property of the null space,
checkable without relying on either library function being correct.

</details>

### 🤖 APPLY
5. Consider a two-layer linear network (no activation function) with an $8\to3\to8$
   architecture: $W_1$ of shape $(3,8)$ then $W_2$ of shape $(8,3)$, so the composed map is
   $\mathbf{y}=W_2W_1\mathbf{x}=W_{\text{eff}}\mathbf{x}$ with $W_{\text{eff}}=W_2W_1$ of shape
   $(8,8)$. Compute $\text{rank}(W_{\text{eff}})$ for random $W_1,W_2$, and compare it to what an
   arbitrary (unconstrained) $8\times8$ matrix's rank could be.

<details>
<summary>✅ What you should observe</summary>

An arbitrary $8\times8$ matrix generically has full rank 8, but $\text{rank}(W_{\text{eff}})$
here is capped at $3$ — exactly the hidden dimension — no matter what $W_1,W_2$ are (verified:
`np.linalg.matrix_rank(W2 @ W1)` returns `3` for random Gaussian `W1`, `W2` of the stated shapes,
even though $W_{\text{eff}}$ itself is a "generic-looking" $8\times8$ matrix with no zero entries
that would visually hint at a rank deficiency). Feeding *any* number of different inputs through
this network and stacking the outputs never recovers more than rank $3$ among them either. This
is the direct generalization of the notebook's own rank-2 `W_low_rank` example to a genuine
multi-layer architecture: an intermediate "bottleneck" dimension caps the rank of the whole
composed transformation at $\min$(bottleneck width, input rank, output rank) — the same structural
fact that motivates deliberately narrow bottleneck layers (autoencoders) and low-rank adapters
(LoRA) alike.

</details>

### 🚀 CHALLENGE
6. The notebook's `M` (rank 2, since row 2 = $2\times$ row 1 exactly) is *exactly* rank-deficient
   by construction. Investigate what happens to `np.linalg.matrix_rank` and the null space
   dimension as you add progressively smaller random noise to that dependent row — say noise
   scales $10^{-3}, 10^{-6}, 10^{-9}, 10^{-12}, 10^{-15}$ — instead of leaving it exactly
   dependent. At what noise scale does the reported rank switch from 2 back to 3, and why isn't
   the answer "as soon as the noise is nonzero"?

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer computes the singular values of the perturbed matrix at each noise scale (not
just calling `matrix_rank` as a black box) and observes that the third singular value grows
roughly proportionally with the injected noise (e.g. around $2\times10^{-5}$ at noise scale
$10^{-3}$, shrinking to around $4\times10^{-11}$ at noise scale $10^{-9}$), while the top two
singular values stay essentially fixed at their unperturbed values (around $8.44$ and $0.92$
here). `np.linalg.matrix_rank` doesn't test for exact zero; it counts singular values above a
tolerance that scales with the matrix's largest singular value and its size (`rank_tol ≈
max(singular values) × max(shape) × machine epsilon`, roughly $5.6\times10^{-15}$ for this
matrix). The reported rank switches from 2 to 3 once the injected noise pushes the third
singular value above that threshold — empirically, this happens somewhere between noise scales
of $10^{-13}$ and $10^{-12}$ for this specific matrix, not at literally any nonzero noise. The
broader point: "rank" as computed by any floating-point library is always rank *up to a
tolerance*, which matters directly for real learned weight matrices — a matrix that is
*mathematically* full rank but has one tiny singular value is numerically indistinguishable from
a lower-rank matrix, which is precisely the regime low-rank approximation methods exploit.

</details>

---

## 📚 Further Reading
- Strang, *Introduction to Linear Algebra*, Ch. 3 (The Four Fundamental Subspaces)
- [3Blue1Brown: Essence of Linear Algebra](https://www.3blue1brown.com/topics/linear-algebra) (video series)

---

*Next notebook: [Linear Transformations](03_linear_transformations.ipynb)*